# Open-set refusal on EVA02 OOF (inner CV)

Closed-test candidate mode on identity-disjoint 5-fold OOF of LLM2CLIP EVA02-L-14-336 (`runs/cv/eva02_trial23`). Raw embeddings, no TTA, no postproc. `camera_id` / `vehicle_id` are labels for building pairs only, never features.

**Protocol (nested 5-fold).** For test fold `f`:

1. **Train packs (50/50).** Same query vs full gallery (`y=1`) and vs that identity stripped (`y=0`). CatBoost and TabM fit on these pairs.
2. **Eval / calibration packs (~20% open).** Hold out 20% of query identities from the gallery (`open_set_pack`). One row per query; prevalence matches the hidden-test mix. Thresholds, F1, TNR, and outer-OOF use only these packs.
3. Inner CV on the other four folds: train CatBoost/TabM on three 50/50 packs, score the fourth eval pack. Cosine needs no fit.
4. Freeze each head's contest operating point (`0.7 × F1 + 0.3 × TNR`) on pooled inner **eval** scores. Rank-average weights and the three-head vote also freeze here.
5. Retrain CatBoost/TabM on all four 50/50 packs and score fold `f`'s eval pack with that fold's inner-CV thresholds. Do not reuse a threshold that saw fold `f`.
6. Outer-OOF concatenates those per-fold accept/refuse masks (one micro TP/FP/FN/TN). Serving is the mean inner-CV **cosine** threshold: Cosine won nested contest score, so Docker is `eva02_threshold`, not the two-head AND.

TNR uses only no-match queries. F1 and TNR are **micro**. F1 is query-level: TP requires a gallery match **and** a correct top-1; a closed query with the wrong top-1 is FP. Candidates below rank-1 do not affect F1 or TNR.

| Head | Score |
| --- | --- |
| Cosine | max query–gallery cosine (**Docker serving**) |
| CatBoost | `P(match exists)` |
| serving AND | cosine and CatBoost both accept (`eva02_ensemble`, not Docker) |
| TabM | mean sigmoid over `k` BatchEnsemble members |
| rank-mean | equal-weight rank average of cosine, CatBoost, TabM |
| rank-weighted | inner-CV nonnegative weights on those ranks |
| majority | accept if at least two of the three heads accept |
| unanimous | accept only if all three heads accept |

Contest metrics: **`0.7 × F1 + 0.3 × TNR` at the frozen inner-CV threshold**, plus F1, TNR, and PR-AUC. Rank-average / TabM / three-head votes stay OOF calibration only.


In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "train.csv").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve

from postproc.retrieval import normalize
from refusal import (
    OPEN_SET_FRACTION,
    STAT_NAMES,
    balanced_pack,
    candidate_metrics,
    decide,
    decision_metrics,
    feature_names,
    fit_boosting,
    fit_rank_weights,
    fit_tabm,
    open_set_pack,
    open_set_split,
    predict_boosting,
    predict_tabm,
    rank_average,
    ranking_metrics,
    select_threshold,
    similarities,
    sweep_thresholds,
    vote_fraction,
)

plt.rcParams.update(
    {
        "figure.figsize": (10, 4),
        "axes.grid": True,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
pd.set_option("display.max_columns", 24)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)


def display(obj):
    print(obj if isinstance(obj, str) else obj.to_string() if hasattr(obj, "to_string") else obj)


CV = ROOT / "runs" / "cv" / "eva02_trial23"
FIG_DIR = ROOT / "notebooks" / "eva02" / "refusal_analysis_figs"
FIG_DIR.mkdir(parents=True, exist_ok=True)
K_RETR = 10
HEADS = ("cosine", "catboost", "tabm")
print("ROOT", ROOT)
print("CV", CV)

ROOT /data/projects/ryzhichkin/ReID
CV /data/projects/ryzhichkin/ReID/runs/cv/eva02_trial23


## Load OOF folds: 50/50 train packs and 20% identity-hold-out eval packs


In [3]:
def load_pack(directory):
    oof = pd.read_csv(directory / "oof.csv", dtype={"image_id": str})
    query = pd.read_csv(directory / "query.csv", dtype={"image_id": str})
    gallery = pd.read_csv(directory / "gallery.csv", dtype={"image_id": str})
    emb = normalize(np.load(directory / "embeddings.npy"))
    lookup = {image_id: i for i, image_id in enumerate(oof.image_id)}
    qe = emb[[lookup[i] for i in query.image_id]]
    ge = emb[[lookup[i] for i in gallery.image_id]]
    return {"query": query, "gallery": gallery, "qe": qe, "ge": ge}


FOLDS = {fold: load_pack(CV / f"fold{fold}" / "val") for fold in range(5)}
TRAIN = {}
EVAL = {}
for fold, pack in FOLDS.items():
    qids = pack["query"].vehicle_id.to_numpy()
    gids = pack["gallery"].vehicle_id.to_numpy()
    X, y, cos, top = balanced_pack(
        pack["qe"],
        pack["ge"],
        qids,
        gids,
        k=K_RETR,
        with_embeddings=True,
    )
    eX, ey, ecos, etop, held = open_set_pack(
        pack["qe"],
        pack["ge"],
        qids,
        gids,
        k=K_RETR,
        with_embeddings=True,
        fraction=OPEN_SET_FRACTION,
        seed=fold,
    )
    TRAIN[fold] = {"X": X, "y": y, "cos": cos, "top": top}
    EVAL[fold] = {"X": eX, "y": ey, "cos": ecos, "top": etop, "held": held}
    print(
        fold,
        "train",
        len(y),
        "pos",
        float(y.mean()),
        "eval",
        len(ey),
        "open",
        float(1.0 - ey.mean()),
        "held",
        len(held),
        "top1",
        float(etop[ey.astype(bool)].mean()) if ey.any() else float("nan"),
        "dim",
        X.shape[1],
    )

0 train 618 pos 0.5 eval 309 open 0.20064724919093846 held 62 top1 0.854251012145749 dim 535


1 train 616 pos 0.5 eval 308 open 0.2012987012987013 held 62 top1 0.8414634146341463 dim 535


2 train 616 pos 0.5 eval 308 open 0.2012987012987013 held 62 top1 0.8455284552845529 dim 535


3 train 616 pos 0.5 eval 308 open 0.2012987012987013 held 62 top1 0.8617886178861789 dim 535


4 train 616 pos 0.5 eval 308 open 0.2012987012987013 held 62 top1 0.8821138211382114 dim 535


## Nested 5-fold fit

Inner models see only three of the four training folds (50/50 packs). Thresholds, blend weights, and the serving AND rule freeze on inner **eval** packs and never touch the outer test fold. Each outer fold keeps its own inner-CV accept/refuse mask. `serving_and` applies that fold's cosine and CatBoost thresholds jointly: accept if both heads accept.


In [4]:
def concat_folds(store, folds, key):
    return np.concatenate([store[fold][key] for fold in folds], 0)


def fit_models(train_folds, seed, epochs):
    X, y = concat_folds(TRAIN, train_folds, "X"), concat_folds(TRAIN, train_folds, "y")
    cat = fit_boosting(X, y, iterations=200, depth=4, learning_rate=0.08, seed=seed)
    tabm = fit_tabm(
        X,
        y,
        epochs=epochs,
        batch_size=128,
        patience=6,
        hidden=(64, 32),
        dropout=0.1,
        k=8,
        seed=seed,
        val_frac=0.15,
    )
    return cat, tabm


def predict_fold(cat, tabm, fold):
    return {
        "y": EVAL[fold]["y"],
        "top": EVAL[fold]["top"],
        "cosine": EVAL[fold]["cos"],
        "catboost": predict_boosting(cat, EVAL[fold]["X"]),
        "tabm": predict_tabm(tabm, EVAL[fold]["X"]),
    }


def summarize(name, y, scores, threshold, top):
    return {"method": name, **candidate_metrics(y, scores, threshold, top), **ranking_metrics(y, scores)}


ROWS = []
OOF: dict[str, dict[str, Any]] = {
    name: {"y": [], "top": [], "scores": [], "threshold": [], "accept": []}
    for name in (
        "always_accept",
        "cosine",
        "catboost",
        "serving_and",
        "tabm",
        "rank_mean",
        "rank_weighted",
        "majority",
        "unanimous",
    )
}
BLEND_W = []
MODELS: dict[int, dict[str, Any]] = {}
for test_fold in range(5):
    others = [fold for fold in range(5) if fold != test_fold]
    inner_y = []
    inner_top = []
    inner = {name: [] for name in HEADS}
    for val_fold in others:
        train_inner = [fold for fold in others if fold != val_fold]
        cat_i, tabm_i = fit_models(train_inner, seed=10 * test_fold + val_fold, epochs=12)
        pred_i = predict_fold(cat_i, tabm_i, val_fold)
        inner_y.append(pred_i["y"])
        inner_top.append(pred_i["top"])
        for name in HEADS:
            inner[name].append(pred_i[name])
    y_in = np.concatenate(inner_y)
    top_in = np.concatenate(inner_top)
    s_in = {name: np.concatenate(inner[name]) for name in HEADS}
    t_head = {name: select_threshold(y_in, s_in[name], top_in, kind="contest") for name in HEADS}
    channels_in = [s_in[name] for name in HEADS]
    t_rank = select_threshold(y_in, rank_average(channels_in), top_in, kind="contest")
    weights, t_weighted = fit_rank_weights(y_in, channels_in, top_in, grid=4)
    head_t = [t_head[name]["threshold"] for name in HEADS]
    votes_in = vote_fraction(channels_in, head_t)
    t_vote = select_threshold(y_in, votes_in, top_in, kind="contest")

    cat, tabm = fit_models(others, seed=test_fold, epochs=16)
    ev = predict_fold(cat, tabm, test_fold)
    channels = [ev[name] for name in HEADS]
    rank = rank_average(channels)
    rank_w = rank_average(channels, weights=weights)
    votes = vote_fraction(channels, head_t)
    methods = {
        "always_accept": (ev["cosine"], -1.0, decide(ev["cosine"], -1.0)),
        "cosine": (ev["cosine"], t_head["cosine"]["threshold"], None),
        "catboost": (ev["catboost"], t_head["catboost"]["threshold"], None),
        "tabm": (ev["tabm"], t_head["tabm"]["threshold"], None),
        "rank_mean": (rank, t_rank["threshold"], None),
        "rank_weighted": (rank_w, t_weighted["threshold"], None),
        "majority": (votes, 2.0 / 3.0, None),
        "unanimous": (votes, 1.0, None),
    }
    serving_accept = decide(ev["cosine"], t_head["cosine"]["threshold"]) & decide(
        ev["catboost"], t_head["catboost"]["threshold"]
    )
    methods["serving_and"] = (np.minimum(ev["cosine"], ev["catboost"]), float("nan"), serving_accept)
    print(
        f"fold {test_fold}: eval {len(ev['y'])} open {float(1.0 - ev['y'].mean()):.3f} "
        f"inner {len(y_in)} weights {np.round(weights, 3)} vote-t {t_vote['threshold']:.2f}"
    )
    MODELS[test_fold] = {"catboost": cat, "tabm": tabm, "eval": ev, "t_head": t_head, "weights": weights}
    BLEND_W.append(weights)
    for name, (pred, thr, accept) in methods.items():
        mask = decide(pred, thr) if accept is None else accept
        row = {
            "method": name,
            **decision_metrics(ev["y"].astype(bool), mask, ev["top"]),
            **ranking_metrics(ev["y"], pred),
            "threshold": thr,
        }
        row["fold"] = test_fold
        ROWS.append(row)
        OOF[name]["y"].append(ev["y"])
        OOF[name]["top"].append(ev["top"])
        OOF[name]["scores"].append(pred)
        OOF[name]["threshold"].append(thr)
        OOF[name]["accept"].append(mask)

for name in OOF:
    OOF[name]["y"] = np.concatenate(OOF[name]["y"])
    OOF[name]["top"] = np.concatenate(OOF[name]["top"])
    OOF[name]["scores"] = np.concatenate(OOF[name]["scores"])
    OOF[name]["accept"] = np.concatenate(OOF[name]["accept"])

table = pd.DataFrame(ROWS)
cols = [
    "fold",
    "method",
    "threshold",
    "contest",
    "f1",
    "precision",
    "recall",
    "tnr",
    "fp_closed",
    "pr_auc",
    "n_open",
    "n_closed",
]
display(table[cols])

fold 0: eval 309 open 0.201 inner 1232 weights [1. 0. 0.] vote-t 0.67


fold 1: eval 308 open 0.201 inner 1233 weights [1. 0. 0.] vote-t 0.67


fold 2: eval 308 open 0.201 inner 1233 weights [0.75 0.   0.25] vote-t 0.67


fold 3: eval 308 open 0.201 inner 1233 weights [0.667 0.333 0.   ] vote-t 0.67


fold 4: eval 308 open 0.201 inner 1233 weights [1. 0. 0.] vote-t 0.67
    fold         method  threshold  contest     f1  precision  recall    tnr  fp_closed  pr_auc  n_open  n_closed
0      0  always_accept    -1.0000   0.5681 0.8115     0.6828  1.0000 0.0000         36  0.9813      62       247
1      0         cosine     0.6612   0.8492 0.8260     0.8683  0.7876 0.9032         21  0.9813      62       247
2      0       catboost     0.5049   0.8453 0.8205     0.8800  0.7686 0.9032         18  0.9828      62       247
3      0           tabm     0.5315   0.8380 0.7893     0.8717  0.7212 0.9516         21  0.9805      62       247
4      0      rank_mean     0.2892   0.8413 0.8423     0.8500  0.8348 0.8387         23  0.9826      62       247
5      0  rank_weighted     0.3054   0.8320 0.8292     0.8505  0.8089 0.8387         22  0.9813      62       247
6      0       majority     0.6667   0.8577 0.8244     0.8800  0.7753 0.9355         20  0.9536      62       247
7      0      unan

## Aggregate contest metrics

Mean ± std over the five outer folds (each fold uses only its inner-CV threshold, on that fold's 20% identity-hold-out pack). Outer-OOF micro metrics concatenate those decisions; they do not re-apply a pooled threshold. Serving is the mean inner-CV **cosine** threshold (best nested contest score). `serving_and/oof` is cosine AND CatBoost with per-fold inner thresholds, kept as a comparison. Always-accept TNR is 0. F1 penalizes a closed query whose top-1 identity is wrong.


In [5]:
order = [
    "always_accept",
    "cosine",
    "catboost",
    "serving_and",
    "tabm",
    "rank_mean",
    "rank_weighted",
    "majority",
    "unanimous",
]
agg = (
    table.groupby("method")[["contest", "f1", "precision", "recall", "tnr", "pr_auc", "threshold"]]
    .agg(["mean", "std"])
    .loc[order]
)
display(agg)

pooled_rows = []
deploy_rows = []
for name in order:
    y = OOF[name]["y"]
    top = OOF[name]["top"]
    scores = OOF[name]["scores"]
    row = {
        "method": name + "/oof",
        **decision_metrics(y, OOF[name]["accept"], top),
        **ranking_metrics(y, scores),
    }
    pooled_rows.append(row)
    deploy_t = float(np.nanmean(OOF[name]["threshold"]))
    if name == "serving_and":
        cos_t = float(np.mean(OOF["cosine"]["threshold"]))
        cb_t = float(np.mean(OOF["catboost"]["threshold"]))
        deploy_accept = decide(OOF["cosine"]["scores"], cos_t) & decide(OOF["catboost"]["scores"], cb_t)
        deploy = {
            "method": name + "/deploy_t",
            **decision_metrics(y, deploy_accept, top),
            **ranking_metrics(y, scores),
            "threshold": float("nan"),
        }
    else:
        deploy = summarize(name + "/deploy_t", y, scores, deploy_t, top)
    deploy_rows.append(deploy)
pooled = pd.DataFrame(pooled_rows)
deploy = pd.DataFrame(deploy_rows)
display(
    pooled[
        ["method", "contest", "f1", "precision", "recall", "tnr", "fp_closed", "pr_auc", "n_open", "n_closed"]
    ]
)
display(deploy[["method", "threshold", "contest", "f1", "tnr"]])
print("blend weights mean", np.round(np.mean(BLEND_W, 0), 3), "std", np.round(np.std(BLEND_W, 0), 3))
print(
    "serving cosine",
    round(float(np.mean(OOF["cosine"]["threshold"])), 4),
    "catboost",
    round(float(np.mean(OOF["catboost"]["threshold"])), 4),
)

              contest            f1        precision        recall           tnr        pr_auc        threshold       
                 mean    std   mean    std      mean    std   mean    std   mean    std   mean    std      mean    std
method                                                                                                                
always_accept  0.5689 0.0063 0.8127 0.0090    0.6846 0.0128 1.0000 0.0000 0.0000 0.0000 0.9702 0.0080   -1.0000 0.0000
cosine         0.8240 0.0205 0.8219 0.0308    0.8599 0.0258 0.7918 0.0788 0.8290 0.1150 0.9702 0.0080    0.6638 0.0015
catboost       0.8154 0.0200 0.8068 0.0402    0.8682 0.0296 0.7594 0.0926 0.8355 0.1072 0.9657 0.0113    0.5540 0.0317
serving_and    0.8197 0.0199 0.8046 0.0401    0.8723 0.0283 0.7524 0.0910 0.8548 0.1020 0.9694 0.0088       NaN    NaN
tabm           0.8067 0.0372 0.8152 0.0294    0.8437 0.0383 0.7957 0.0910 0.7871 0.1779 0.9668 0.0082    0.4673 0.0451
rank_mean      0.8335 0.0057 0.8341 0.0086    0.

/tmp/ipykernel_71753/2195580495.py:31: RuntimeWarning: Mean of empty slice
  deploy_t = float(np.nanmean(OOF[name]["threshold"]))


## Inner-CV cosine threshold (fold 0)

The sweep below is the inner OOF of fold 0's training folds **eval packs** (20% identity hold-out). The vertical line is the frozen contest point (`0.7 × F1 + 0.3 × TNR`) applied to fold 0's eval pack. F1 uses contest top-1 identity, not pair-level hits. max-F1 / Youden / F1×TNR are shown only as alternate rules.


In [6]:
fold0_others = [1, 2, 3, 4]
inner_y = []
inner_cos = []
inner_top = []
for val_fold in fold0_others:
    inner_y.append(EVAL[val_fold]["y"])
    inner_cos.append(EVAL[val_fold]["cos"])
    inner_top.append(EVAL[val_fold]["top"])
y_in = np.concatenate(inner_y)
cos_in = np.concatenate(inner_cos)
top_in = np.concatenate(inner_top)
y0 = EVAL[0]["y"]
cos0 = EVAL[0]["cos"]
top0 = EVAL[0]["top"]
rules = {
    "contest": select_threshold(y_in, cos_in, top_in, kind="contest"),
    "max_f1": select_threshold(y_in, cos_in, top_in, kind="max_f1"),
    "youden": select_threshold(y_in, cos_in, top_in, kind="youden"),
    "f1_tnr": select_threshold(y_in, cos_in, top_in, kind="f1_tnr"),
}
rule_tab = []
for name, picked in rules.items():
    row = candidate_metrics(y0, cos0, picked["threshold"], top0)
    row["rule"] = name
    row["cal_contest"] = picked["contest"]
    row["cal_f1"] = picked["f1"]
    row["cal_tnr"] = picked["tnr"]
    rule_tab.append(row)
display(
    pd.DataFrame(rule_tab)[
        [
            "rule",
            "threshold",
            "cal_contest",
            "cal_f1",
            "cal_tnr",
            "contest",
            "f1",
            "tnr",
            "precision",
            "recall",
            "fp_closed",
        ]
    ]
)

sweep_cal = pd.DataFrame(sweep_thresholds(y_in, cos_in, top_in))
sweep_ev = pd.DataFrame(sweep_thresholds(y0, cos0, top0))
fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
ax[0].plot(sweep_cal.threshold, sweep_cal.contest, label="0.7 F1 + 0.3 TNR")
ax[0].plot(sweep_cal.threshold, sweep_cal.f1, label="F1")
ax[0].plot(sweep_cal.threshold, sweep_cal.tnr, label="TNR")
ax[0].axvline(rules["contest"]["threshold"], color="C3", ls="--", label="contest")
ax[0].set_title("inner CV (folds 1–4)")
ax[0].set_xlabel("cosine threshold")
ax[0].legend()
ax[1].plot(sweep_ev.threshold, sweep_ev.contest, label="0.7 F1 + 0.3 TNR")
ax[1].plot(sweep_ev.threshold, sweep_ev.f1, label="F1")
ax[1].plot(sweep_ev.threshold, sweep_ev.tnr, label="TNR")
ax[1].axvline(rules["contest"]["threshold"], color="C3", ls="--", label="frozen")
ax[1].set_title("outer fold 0, 20% identity hold-out")
ax[1].set_xlabel("cosine threshold")
ax[1].legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "threshold_sweep.png", dpi=140, bbox_inches="tight")
plt.show()

      rule  threshold  cal_contest  cal_f1  cal_tnr  contest     f1    tnr  precision  recall  fp_closed
0  contest     0.6612       0.8229  0.8299   0.8065   0.8492 0.8260 0.9032     0.8683  0.7876         21
1   max_f1     0.5472       0.7543  0.8546   0.5202   0.7696 0.8505 0.5806     0.7860  0.9266         29
2   youden     0.6612       0.8229  0.8299   0.8065   0.8492 0.8260 0.9032     0.8683  0.7876         21
3   f1_tnr     0.6867       0.8174  0.8030   0.8508   0.8482 0.8038 0.9516     0.8936  0.7304         17


Saved figure:

![threshold sweep](refusal_analysis_figs/threshold_sweep.png)


## Score separation on pooled OOF pairs


In [7]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, name, title in zip(axes, HEADS, ("max cosine", "CatBoost", "TabM"), strict=True):
    y = OOF[name]["y"].astype(bool)
    s = OOF[name]["scores"]
    ax.hist(s[y], bins=30, alpha=0.7, label="has match", density=True)
    ax.hist(s[~y], bins=30, alpha=0.7, label="no match", density=True)
    ax.axvline(np.mean(OOF[name]["threshold"]), color="k", ls="--", label="deploy t")
    ax.set_title(title)
    ax.set_xlabel("score")
axes[0].set_ylabel("density")
axes[0].legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "score_hist.png", dpi=140, bbox_inches="tight")
plt.show()

Saved figure:

![score histograms](refusal_analysis_figs/score_hist.png)


## Precision–recall, including ensembles


In [8]:
fig, ax = plt.subplots(figsize=(6.5, 5.2))
for name, title in (
    ("cosine", "cosine"),
    ("catboost", "CatBoost"),
    ("serving_and", "serving AND"),
    ("tabm", "TabM"),
    ("rank_mean", "rank-mean"),
    ("rank_weighted", "rank-weighted"),
    ("majority", "majority votes"),
):
    y = OOF[name]["y"]
    s = OOF[name]["scores"]
    p, r, _ = precision_recall_curve(y, s)
    ap = ranking_metrics(y, s)["pr_auc"]
    ax.plot(r, p, label=f"{title}  PR-AUC={ap:.3f}")
ax.set_xlabel("recall (has-match)")
ax.set_ylabel("precision")
ax.set_title("Concatenated outer-OOF PR curves")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "pr_curves.png", dpi=140, bbox_inches="tight")
plt.show()

Saved figure:

![PR curves](refusal_analysis_figs/pr_curves.png)


## Head-to-head, with ensembles last


In [9]:
fig, ax = plt.subplots(figsize=(11.5, 4.2))
methods = order
labels = [
    "always accept",
    "cosine t",
    "CatBoost",
    "serving AND",
    "TabM",
    "rank-mean",
    "rank-weighted",
    "majority",
    "unanimous",
]
x = np.arange(len(methods))
width = 0.2
means = table.groupby("method")[["contest", "f1", "tnr", "pr_auc"]].mean().loc[methods]
for i, metric in enumerate(("contest", "f1", "tnr", "pr_auc")):
    label = {"contest": "0.7 F1+0.3 TNR", "f1": "F1", "tnr": "TNR", "pr_auc": "PR-AUC"}[metric]
    ax.bar(x + (i - 1.5) * width, means[metric], width, label=label)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("5-fold mean")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "metrics_bars.png", dpi=140, bbox_inches="tight")
plt.show()

Saved figure:

![metrics bars](refusal_analysis_figs/metrics_bars.png)


## CatBoost statistics


In [10]:
names = feature_names(FOLDS[0]["qe"].shape[1])
imps = np.stack([MODELS[fold]["catboost"].get_feature_importance() for fold in range(5)]).mean(0)
imp = pd.DataFrame({"feature": names, "importance": imps}).sort_values("importance", ascending=False)
display(imp.head(15))
stat_imp = imp[imp.feature.isin(STAT_NAMES)].copy()
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(stat_imp.feature[::-1], stat_imp.importance[::-1])
ax.set_title("CatBoost mean importance, retrieval statistics")
fig.tight_layout()
fig.savefig(FIG_DIR / "catboost_stats.png", dpi=140, bbox_inches="tight")
plt.show()

          feature  importance
0            top1     17.8751
9       mean_top3      4.4955
1            top2      4.0139
13   pairwise_std      3.2504
6           std_k      3.1965
11        entropy      3.0831
22    margin_rest      2.7342
10      mean_top5      2.6542
16        n_ge_07      2.5680
14  query_to_mean      1.1919
5          mean_k      1.1657
2            top3      1.0022
12  pairwise_mean      0.8578
7        median_k      0.7570
47           q_24      0.6341


Saved figure:

![CatBoost stats](refusal_analysis_figs/catboost_stats.png)


## Why the cosine threshold is 0.66

ArcFace (`m = 0.479` rad ≈ 27.4°, `s = 35.1`) and AdaSP (`τ = 0.026`, `1/τ ≈ 38`) do **not** algebraically fix an accept cut. AdaSP on PK 16×2 sees one positive pair per identity and ~30 in-batch negatives; it enlarges the *gap* between a true match and a hard negative. Retrieval max-impostor is an extreme value over ~870 gallery crops, not those 30.

On the same 20% identity-hold-out packs: plot max cosine for closed queries vs open queries, the true-match max, and the best impostor. The contest cut is the midpoint of the two max-cosine distributions.


In [11]:
ARCFACE_M = 0.4789452160852028
ADASP_TAU = 0.026071235242105424
serving_t = float(np.mean(OOF["cosine"]["threshold"]))
closed_max, closed_true, closed_impostor, open_max, intra = [], [], [], [], []
for fold, pack in FOLDS.items():
    qids = pack["query"].vehicle_id.to_numpy()
    gids = pack["gallery"].vehicle_id.to_numpy()
    sim = similarities(pack["qe"], pack["ge"])
    _, keep, _held = open_set_split(qids, gids, fraction=OPEN_SET_FRACTION, seed=fold)
    sim_e = sim[:, keep]
    gids_e = gids[keep]
    for i in range(len(qids)):
        row = sim_e[i]
        same = gids_e == qids[i]
        if same.any():
            closed_max.append(float(row.max()))
            closed_true.append(float(row[same].max()))
            closed_impostor.append(float(row[~same].max()))
            intra.extend(row[same].astype(np.float64).tolist())
        else:
            open_max.append(float(row.max()))
closed_max = np.asarray(closed_max)
closed_true = np.asarray(closed_true)
closed_impostor = np.asarray(closed_impostor)
open_max = np.asarray(open_max)
intra = np.asarray(intra)
gap = closed_true - closed_impostor
display(
    pd.DataFrame(
        {
            "mean": [
                closed_max.mean(),
                closed_true.mean(),
                closed_impostor.mean(),
                open_max.mean(),
                intra.mean(),
            ],
            "median": [
                np.median(closed_max),
                np.median(closed_true),
                np.median(closed_impostor),
                np.median(open_max),
                np.median(intra),
            ],
            "p25": [
                np.quantile(closed_max, 0.25),
                np.quantile(closed_true, 0.25),
                np.quantile(closed_impostor, 0.25),
                np.quantile(open_max, 0.25),
                np.quantile(intra, 0.25),
            ],
        },
        index=pd.Index(["closed max", "true-match max", "impostor max", "open max", "intra pair"]),
    )
)
print(
    f"serving t {serving_t:.4f}  median midpoint {(np.median(closed_max) + np.median(open_max)) / 2:.4f}  "
    f"mean midpoint {(closed_max.mean() + open_max.mean()) / 2:.4f}"
)
print(
    f"gap mean {gap.mean():.3f}  median {np.median(gap):.3f}  gap/tau {gap.mean() / ADASP_TAU:.1f}  "
    f"P(gap<0) {float((gap < 0).mean()):.3f}"
)
print(
    f"ArcFace m {np.degrees(ARCFACE_M):.1f} deg  sqrt(E intra) {float(np.sqrt(intra.mean())):.3f}  "
    f"cos(theta+m) {float(np.cos(np.arccos(np.clip(np.sqrt(intra.mean()), 0, 1)) + ARCFACE_M)):.3f}"
)
fig, ax = plt.subplots(figsize=(7.2, 4.0))
ax.hist(closed_max, bins=40, density=True, alpha=0.65, label="closed max cosine")
ax.hist(open_max, bins=40, density=True, alpha=0.65, label="open max cosine")
ax.axvline(serving_t, color="C3", ls="--", label=f"serving t={serving_t:.3f}")
ax.axvline(
    (np.median(closed_max) + np.median(open_max)) / 2,
    color="0.35",
    ls=":",
    label="median midpoint",
)
ax.set_xlabel("max query–gallery cosine")
ax.set_ylabel("density")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "cosine_threshold_geometry.png", dpi=140, bbox_inches="tight")
plt.show()

                 mean  median    p25
closed max     0.7615  0.7802 0.6908
true-match max 0.7435  0.7745 0.6726
impostor max   0.5462  0.5325 0.4566
open max       0.5468  0.5324 0.4517
intra pair     0.6902  0.7224 0.6000
serving t 0.6638  median midpoint 0.6563  mean midpoint 0.6541
gap mean 0.197  median 0.212  gap/tau 7.6  P(gap<0) 0.143
ArcFace m 27.4 deg  sqrt(E intra) 0.831  cos(theta+m) 0.481


Saved figure:

![cosine threshold geometry](refusal_analysis_figs/cosine_threshold_geometry.png)


## Takeaway

- CatBoost and TabM still train on 50/50 stripped-identity pairs. Thresholds, F1, TNR, and outer-OOF use a separate 20% identity-hold-out pack (`open_set_pack`), matching hidden-test prevalence.
- The operating point maximizes the contest score `0.7 × F1 + 0.3 × TNR` on inner-CV **eval** scores. max-F1 alone can pick a point with TNR 0.
- Contest F1 is query-level: TP = has match and top-1 identity is correct. Extra candidates do not count as extra FP. Closed wrong top-1 is FP (`fp_closed`).
- TNR uses only no-match queries. Always-accept still has TNR 0.
- Outer-OOF accept/refuse uses each fold's own inner-CV threshold. Serving cosine/CatBoost thresholds are the mean of those five inner points, not a threshold fit on the evaluated queries.
- **`serving_and` is the Docker rule:** accept if frozen cosine AND CatBoost both accept that query. Rank-average / majority / unanimous combine cosine, CatBoost, and TabM after each head is calibrated on inner CV; those three-head blends are OOF calibration only.
- Hidden-test prevalence is ~20% open. Local eval packs hold out 20% of identities from the gallery, so the reported F1/TNR are not the old 50/50 probe.
